### Images into text, but how?

There's plenty of tutorials on [Object Character Recognition (OCR)](https://www.tecmint.com/python-text-extraction-from-images/), when you have an image of text and then you want the text. I won't add one here, but I will help you turn images into text. How? 

We'll use an image-to-text library to describe the images as best as the model can. This won't be 100% accurate and will be missing all the wonderful extra context we know when we look at an image we took or drew, but it will give us a starting point for our personal data and AI/ML projects.

#### Easiest way

The easiest way is to run the Hugging Face model and pipeline directly on your machine with the GPU.

To do so, make sure you:

1. Have a Python environment set up with required libraries (torch, huggingface transformers).
2. Download the model with Hugging Face CLI first: huggingface-cli download MODEL_PATH
3. Either install and start a Jupyter server there and then navigate in your browser to the [IP:PORT] of the other machine, or SSH in and run the below code in a python shell or file.

You can technically look through and try out any of the [image-to-text models](https://huggingface.co/tasks/image-to-text). Here I've just chosen the Salesforce one based on BLIP (Bootstrapped Language-Image Pretraining) model that is then finetuned to provide image captions.

In [ ]:
from transformers import pipeline

captioner = pipeline("image-to-text", model="Salesforce/blip-image-captioning-base")

In [ ]:
captioner("https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/parrots.png")
## [{'generated_text': 'two birds are standing next to each other '}]

In [ ]:
captioner("PATH_TO_PHOTO_OF_YOURS_HERE")


### Using vLLM with a vision-language model

For full setup of vllm, check out the other notebook in this folder on vLLM. Once you have that running, you can:

- Download one of the vision language models, for an example, I'll use [Qwen/Qwen2-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct)
- start it using the vllm serve command
- Then access it from your laptop using the following code.

In [33]:
from openai import OpenAI

In [34]:
openai_api_key = "EMPTY"
openai_api_base = "http://192.168.1.81:8000/v1" # change this for your setup!

In [35]:
messages = [{
				"role": "user",
				"content": [
					{
						"type": "text",
						"text": "Describe this image in one sentence."
					},
					{
						"type": "image_url",
						"image_url": {
							"url": "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
						}
					}
				]
}]

In [36]:
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

In [37]:
completion = client.chat.completions.create(
    model="Qwen/Qwen2-VL-2B-Instruct",
    messages=messages
)

In [38]:
completion

ChatCompletion(id='chatcmpl-89f3adce14139628', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The Statue of Liberty stands majestically on Liberty Island, with the New York City skyline in the background.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning=None, reasoning_content=None), stop_reason=None, token_ids=None)], created=1769166817, model='Qwen/Qwen2-VL-2B-Instruct', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=23, prompt_tokens=2194, total_tokens=2217, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None, prompt_token_ids=None, kv_transfer_params=None)

In [39]:
import base64

In [40]:
ls ../../../Desktop/maryam_mirzakhani_stanford.jpg

../../../Desktop/maryam_mirzakhani_stanford.jpg


Read about [Maryam Mirzakhani's inspiring work](https://en.wikipedia.org/wiki/Maryam_Mirzakhani). Photo from [Stanford University](https://news.stanford.edu/stories/2017/07/maryam-mirzakhani-stanford-mathematician-and-fields-medal-winner-dies).

In [41]:
img_path = "../../../Desktop/maryam_mirzakhani_stanford.jpg"

In [42]:
with open(img_path, 'rb') as img_file:
    img_data = img_file.read()
    img_base64 = base64.b64encode(img_data).decode('utf-8')

In [43]:
messages = [{
				"role": "user",
				"content": [
					{
						"type": "text",
						"text": "Describe this image in one sentence."
					},
					{
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{img_base64}"
                        },
                    }
				]
}]

In [44]:
completion = client.chat.completions.create(
    model="Qwen/Qwen2-VL-2B-Instruct",
    messages=messages
)

In [45]:
completion.choices[0].message.content

'A woman with short hair wearing a blue jacket is looking off into the distance.'

### Not yet working but hopefully one day: transformers-cli serve

I'm awaiting to see if [HuggingFace Transformers serve command](https://huggingface.co/docs/transformers/main/serving) will eventually support a non-chat interface. Here's a little bit of notes for myself for later. It won't work yet... but I can dream. :)


In [ ]:
import requests

In [ ]:
salesforce_blip_endpoint = 'http://192.168.1.81:8888'

In [ ]:
image_to_caption = {
    "model": "Salesforce/blip-image-captioning-base",
    "input": "https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/parrots.png"
  }

In [ ]:
r = requests.post(salesforce_blip_endpoint,
                  json=image_to_caption, 
                  headers={'Content-Type': "application/json"})